# Regular Expressions for Bioinformatics

## Learning Objectives
- Master the `re` module: `match`, `search`, `findall`, `finditer`, `sub`, `compile`
- Understand regex patterns: character classes, quantifiers, groups, lookahead/lookbehind
- Use flags: `IGNORECASE`, `MULTILINE`, `DOTALL`, `VERBOSE`
- Apply regex to real bioinformatics tasks: restriction sites, gene IDs, protein motifs, PROSITE patterns

**Why this matters:** Biological data is text-heavy -- FASTA headers, GenBank annotations, protein motifs, and sequence patterns all require sophisticated text matching. Regular expressions are the standard tool for this work.

---

[← Previous: Iterators and Generators for Bioinformatics](../../Tier_1_Python_for_Bioinformatics/11_Iterators_and_Generators/01_iterators_and_generators.ipynb) | [Next: Module 13: Classes and Object-Oriented Programming →](../../Tier_1_Python_for_Bioinformatics/13_Classes_and_OOP/01_classes_and_oop.ipynb)

---

In [ ]:
import re

---
## 1. Basic Pattern Matching

The `re` module provides several core functions:

| Function | Description |
|---|---|
| `re.search(pattern, string)` | Find first match **anywhere** in string |
| `re.match(pattern, string)` | Match only at the **beginning** of string |
| `re.fullmatch(pattern, string)` | Match the **entire** string |
| `re.findall(pattern, string)` | Return all matches as a **list of strings** |
| `re.finditer(pattern, string)` | Return all matches as an **iterator of match objects** |
| `re.sub(pattern, repl, string)` | **Replace** all matches |
| `re.split(pattern, string)` | **Split** string by pattern |

In [ ]:
# re.search() -- find first match anywhere in the string
dna = "GCTATGCGATCGATCGTAA"

match = re.search('ATG', dna)
if match:
    print(f"Found '{match.group()}' at position {match.start()}-{match.end()}")

In [ ]:
# re.match() -- matches only at the BEGINNING of the string
dna = "ATGCGATCGATCG"

print(f"match('ATG', ...): {re.match('ATG', dna)}")   # Match object (starts at 0)
print(f"match('CGA', ...): {re.match('CGA', dna)}")   # None (CGA not at start)

In [ ]:
# re.findall() -- return all non-overlapping matches as a list
dna = "ATGCGATCGATGCGATGCTAA"

starts = re.findall('ATG', dna)
print(f"Start codons found: {starts}")
print(f"Count: {len(starts)}")

In [ ]:
# re.finditer() -- returns an iterator of match objects (preferred for positions)
dna = "ATGCGATCGATGCGATGCTAA"

print("All ATG positions:")
for match in re.finditer('ATG', dna):
    print(f"  Position {match.start()}: {match.group()}")

In [ ]:
# Match objects carry useful information
dna = "GCTATGAAACGATCGTAA"
match = re.search('ATG', dna)

print(f"Matched text:  {match.group()}")
print(f"Start position: {match.start()}")
print(f"End position:   {match.end()}")
print(f"Span:           {match.span()}")

---
## 2. Metacharacters and Character Classes

Special characters that have meaning in regex patterns:

| Symbol | Meaning |
|---|---|
| `.` | Any single character (except newline by default) |
| `^` | Start of string |
| `$` | End of string |
| `[ABC]` | Any one character from the set |
| `[^ABC]` | Any character NOT in the set |
| `[A-Z]` | Any character in the range |
| `\d` | Any digit (0-9) |
| `\w` | Any word character (letter, digit, underscore) |
| `\s` | Any whitespace |
| `\b` | Word boundary |
| `\|` | Alternation (OR) |

In [ ]:
# . (dot) -- matches any single character
dna = "ATGCGATCG"

# Find A followed by any character, then G
print(f"Pattern A.G matches: {re.findall('A.G', dna)}")

In [ ]:
# ^ and $ -- anchors for start and end of string
dna = "ATGCGATCGTAA"

print(f"Starts with ATG: {bool(re.match('^ATG', dna))}")
print(f"Ends with TAA:   {bool(re.search('TAA$', dna))}")
print(f"Ends with ATG:   {bool(re.search('ATG$', dna))}")

In [ ]:
# [] -- character classes
dna = "ATGCGATCGATCG"

# Purines (A or G)
purines = re.findall('[AG]', dna)
print(f"Purines: {purines} (count: {len(purines)})")

# Pyrimidines (C or T)
pyrimidines = re.findall('[CT]', dna)
print(f"Pyrimidines: {pyrimidines} (count: {len(pyrimidines)})")

In [ ]:
# [^] -- negated character class
dna = "ATGCGATNNCGATXCG"

# Find non-standard nucleotides (not A, T, G, C)
non_standard = re.findall('[^ATGC]', dna)
print(f"Non-standard characters: {non_standard}")

# Find their positions
for m in re.finditer('[^ATGC]', dna):
    print(f"  Position {m.start()}: '{m.group()}'")

In [ ]:
# | (pipe) -- alternation (OR)
dna = "ATGCGATCGATCGTAGTGATAA"

# Find all stop codons
stop_codons = re.findall('TAA|TAG|TGA', dna)
print(f"Stop codons found: {stop_codons}")

---
## 3. Quantifiers

| Quantifier | Meaning |
|---|---|
| `*` | Zero or more |
| `+` | One or more |
| `?` | Zero or one |
| `{n}` | Exactly n times |
| `{n,m}` | Between n and m times |
| `{n,}` | n or more times |

By default, quantifiers are **greedy** (match as much as possible). Append `?` to make them **lazy** (match as little as possible).

In [ ]:
dna = "AAATTTAAAAAATTTTTGCGCGC"

print(f"A+ (one or more A): {re.findall('A+', dna)}")
print(f"T+ (one or more T): {re.findall('T+', dna)}")
print(f"[GC]+ (GC runs):    {re.findall('[GC]+', dna)}")

In [ ]:
# Exact repetition counts
dna = "ATGATGATGATGATG"

print(f"Triplets (.{{3}}): {re.findall('.{3}', dna)}")
print(f"4-6 chars ([ATGC]{{4,6}}): {re.findall('[ATGC]{4,6}', dna)}")

In [ ]:
# Greedy vs lazy matching -- critical for biological pattern search
dna = "ATGCGATCGTAAATGCCCTAG"

# Greedy: matches the LONGEST possible string
greedy = re.findall('ATG.*TAG', dna)
print(f"Greedy  (ATG.*TAG):  {greedy}")

# Lazy: matches the SHORTEST possible string
lazy = re.findall('ATG.*?TAG', dna)
print(f"Lazy    (ATG.*?TAG): {lazy}")

In [ ]:
# Practical example: find homopolymer runs of 4+ bases
dna = "ATGCAAAAGCCCCCGATTTTTCG"

print(f"Sequence: {dna}")
print("\nHomopolymer runs (4+ bases):")
for m in re.finditer(r'(.)\1{3,}', dna):
    print(f"  '{m.group()}' at position {m.start()} ({len(m.group())} bases)")

---
## 4. Groups

Parentheses `()` create **capturing groups**. They allow you to extract specific parts of a match.

| Syntax | Meaning |
|---|---|
| `(pattern)` | Capturing group |
| `(?:pattern)` | Non-capturing group |
| `(?P<name>pattern)` | Named group |
| `\1`, `\2` | Backreference to group 1, 2, etc. |

In [ ]:
# Capturing groups -- extract parts of a match
header = ">gene1|BRCA1|Homo sapiens breast cancer gene"

match = re.search(r'>([^|]+)\|([^|]+)\|(.+)', header)
if match:
    print(f"Full match:  {match.group(0)}")
    print(f"Gene ID:     {match.group(1)}")
    print(f"Gene name:   {match.group(2)}")
    print(f"Description: {match.group(3)}")

In [ ]:
# Named groups -- (?P<name>pattern)
header = ">sp|P04637|P53_HUMAN Cellular tumor antigen p53 OS=Homo sapiens"

pattern = r'>sp\|(?P<accession>[^|]+)\|(?P<entry_name>\S+)\s+(?P<description>.+?)\s+OS=(?P<organism>.+)'
match = re.search(pattern, header)

if match:
    print(f"Accession:   {match.group('accession')}")
    print(f"Entry name:  {match.group('entry_name')}")
    print(f"Description: {match.group('description')}")
    print(f"Organism:    {match.group('organism')}")

In [ ]:
# Non-capturing groups -- (?:pattern)
# Useful for alternation without capturing
dna = "ATGCGATCGATCGTAGTGATAA"

# With capturing group: findall returns the group, not the full match
with_capture = re.findall('(TAA|TAG|TGA)', dna)
print(f"Capturing group:     {with_capture}")

# With non-capturing group: findall returns the full match
without_capture = re.findall('(?:TAA|TAG|TGA)', dna)
print(f"Non-capturing group: {without_capture}")

In [ ]:
# Backreferences -- find repeated codons (tandem repeats)
dna = "ATGATGATGCATGCATGCATGC"

# Find triplets that repeat at least twice
repeated = re.findall(r'([ATGC]{3})\1+', dna)
print(f"Repeated codons: {repeated}")

# With full context
for m in re.finditer(r'([ATGC]{3})\1+', dna):
    unit = m.group(1)
    copies = len(m.group(0)) // len(unit)
    print(f"  '{unit}' x {copies} at position {m.start()}")

---
## 5. Lookahead and Lookbehind

Lookaround assertions match a position without consuming characters.

| Syntax | Name | Meaning |
|---|---|---|
| `(?=pattern)` | Positive lookahead | Followed by pattern |
| `(?!pattern)` | Negative lookahead | NOT followed by pattern |
| `(?<=pattern)` | Positive lookbehind | Preceded by pattern |
| `(?<!pattern)` | Negative lookbehind | NOT preceded by pattern |

In [ ]:
# Positive lookahead: find ATG only when followed by at least 30 bases before a stop
dna = "ATGCGATCGATCGATCGATCGATCGATCGATCGTAA"

# Find 'G' only when followed by 'C' (G in GC dinucleotide)
gc_dinucs = re.findall('G(?=C)', dna)
print(f"G followed by C: {len(gc_dinucs)} occurrences")

# Find all overlapping 3-mers starting with AT using lookahead
overlapping = re.findall('(?=(AT.))', dna)
print(f"Overlapping 3-mers starting with AT: {overlapping}")

In [ ]:
# Negative lookahead: find a codon that is NOT a stop codon
dna = "ATGAAATTTCCCGGG"
codons = re.findall('.{3}', dna)
print(f"All codons: {codons}")

# Find positions where ATG is NOT followed by a stop codon
non_stop_starts = re.findall('ATG(?!TAA|TAG|TGA)', dna)
print(f"ATG not followed by stop: {non_stop_starts}")

In [ ]:
# Lookbehind: find bases that come after a specific context
dna = "GAATTCGGATCCAAGCTT"

# Find 'C' only when preceded by 'AT' (ATC codons)
after_at = re.findall('(?<=AT)C', dna)
print(f"C preceded by AT: {len(after_at)} occurrences")

# Find bases after EcoRI site (GAATTC)
after_ecori = re.findall('(?<=GAATTC).{3}', dna)
print(f"3 bases after EcoRI site: {after_ecori}")

---
## 6. Substitution and Splitting

In [ ]:
# re.sub() -- find and replace
dna = "ATGCGATCGATCG"

# Transcription: T -> U
rna = re.sub('T', 'U', dna)
print(f"DNA: {dna}")
print(f"RNA: {rna}")

In [ ]:
# Using groups in substitution
# Reformat FASTA headers: add a prefix to gene IDs
fasta = ">gene1 description one\nATGCGATC\n>gene2 description two\nGCTAGCTA"

modified = re.sub(r'>(\w+)', r'>Hsapiens_\1', fasta)
print(modified)

In [ ]:
# Substitution with a function
# Mask long homopolymer runs in a sequence
def mask_repeat(match):
    """Replace long homopolymer runs with N's."""
    seq = match.group(0)
    if len(seq) >= 5:
        return 'N' * len(seq)
    return seq


dna = "ATGAAAAAACGATTTTTTTCGATCG"
masked = re.sub(r'(.)\1{2,}', mask_repeat, dna)
print(f"Original: {dna}")
print(f"Masked:   {masked}")

In [ ]:
# re.split() -- split by pattern
# Split a sequence at restriction sites
dna = "ATGCGAATTCGATCGGATCCATCGAAGCTTATCG"

# Split at any common restriction site
fragments = re.split('GAATTC|GGATCC|AAGCTT', dna)
print(f"Fragments after triple digestion:")
for i, frag in enumerate(fragments, 1):
    print(f"  Fragment {i}: {frag} ({len(frag)} bp)")

---
## 7. Compiled Patterns and Flags

Pre-compile patterns with `re.compile()` for efficiency when reusing the same pattern.

In [ ]:
# Compile a pattern for repeated use
start_codon = re.compile('ATG')
stop_pattern = re.compile('TAA|TAG|TGA')

dna = "ATGCGATCGATGCGTAAATGCCCTAG"

print(f"Start codons: {start_codon.findall(dna)}")
print(f"Stop codons:  {stop_pattern.findall(dna)}")

# Compiled patterns have the same methods: search, match, findall, finditer, sub, split
for m in start_codon.finditer(dna):
    print(f"  ATG at position {m.start()}")

In [ ]:
# re.IGNORECASE (re.I) -- case-insensitive matching
dna_mixed = "atgCGAtcgATGccc"

# Without flag: misses lowercase ATG
print(f"Case sensitive:   {re.findall('ATG', dna_mixed)}")

# With flag: finds both
print(f"Case insensitive: {re.findall('ATG', dna_mixed, re.IGNORECASE)}")

In [ ]:
# re.MULTILINE (re.M) -- ^ and $ match at line boundaries
fasta = """>gene1
ATGCGATCG
>gene2
GCTAGCTAG"""

# Without MULTILINE: ^ only matches start of entire string
print(f"Without MULTILINE: {re.findall('^>.*', fasta)}")

# With MULTILINE: ^ matches start of each line
print(f"With MULTILINE:    {re.findall('^>.*', fasta, re.MULTILINE)}")

In [ ]:
# re.DOTALL (re.S) -- make . match newlines too
genbank_entry = ">gene1\nATGCGA\nTCGATCG"

# Without DOTALL: . does not match \n
print(f"Without DOTALL: {re.findall('>gene1.+', genbank_entry)}")

# With DOTALL: . matches everything including \n
print(f"With DOTALL:    {re.findall('>gene1.+', genbank_entry, re.DOTALL)}")

In [ ]:
# re.VERBOSE (re.X) -- write readable patterns with comments
# Pattern to match a restriction enzyme site with ambiguity
bamhi_like = re.compile(r'''
    G           # First guanine
    [AG]        # A or G (purine = R in IUPAC)
    ATC         # ATC core
    [CT]        # C or T (pyrimidine = Y in IUPAC)
    C           # Final cytosine
''', re.VERBOSE)

dna = "ATGCGATCCGATCCGAATCCGATCGC"
matches = bamhi_like.findall(dna)
print(f"BamHI-like sites found: {matches}")

---
## 8. Bio Application: Restriction Enzyme Sites

In [ ]:
RESTRICTION_ENZYMES = {
    'EcoRI':   'GAATTC',
    'BamHI':   'GGATCC',
    'HindIII': 'AAGCTT',
    'NotI':    'GCGGCCGC',
    'XhoI':    'CTCGAG',
    'SalI':    'GTCGAC',
    'NdeI':    'CATATG',
}


def restriction_map(sequence, enzymes):
    """Find all restriction enzyme cut sites in a DNA sequence."""
    results = {}
    for enzyme, site in enzymes.items():
        positions = [m.start() for m in re.finditer(site, sequence)]
        if positions:
            results[enzyme] = positions
    return results


# Construct a plasmid-like sequence with known sites
plasmid = (
    "ATGCGATCG"
    "GAATTC"      # EcoRI
    "GATCGATCG"
    "GGATCC"      # BamHI
    "GATCG"
    "AAGCTT"      # HindIII
    "GATCGATCG"
    "GAATTC"      # EcoRI (second)
    "GATCGATCG"
)

sites = restriction_map(plasmid, RESTRICTION_ENZYMES)

print(f"Plasmid length: {len(plasmid)} bp\n")
print("Restriction map:")
for enzyme, positions in sorted(sites.items()):
    site = RESTRICTION_ENZYMES[enzyme]
    print(f"  {enzyme:8s} ({site}): {positions}")

In [ ]:
# Predict fragment sizes from a single enzyme digest
def predict_fragments(sequence, site):
    """Predict fragment sizes from restriction digest."""
    fragments = re.split(site, sequence)
    return [len(f) for f in fragments if f]


ecori_fragments = predict_fragments(plasmid, 'GAATTC')
print(f"EcoRI digest fragments: {ecori_fragments} bp")
print(f"Sum: {sum(ecori_fragments)} bp (original: {len(plasmid)} bp, sites: {len(plasmid) - sum(ecori_fragments)} bp lost)")

---
## 9. Bio Application: Extract Gene IDs from FASTA Headers

In [ ]:
def parse_fasta_header(header):
    """Parse various FASTA header formats using regex."""
    patterns = [
        # GenBank: >gi|12345|gb|ABC123.1| description
        (r'>gi\|(?P<gi>\d+)\|(?P<db>\w+)\|(?P<accession>[^|]+)\|\s*(?P<description>.*)',
         'GenBank'),
        # UniProt: >sp|P12345|GENE_SPECIES description OS=organism
        (r'>(?:sp|tr)\|(?P<accession>[^|]+)\|(?P<entry_name>\S+)\s+(?P<description>.+?)\s+OS=(?P<organism>[^=]+?)(?:\s+\w+=|$)',
         'UniProt'),
        # NCBI RefSeq: >NM_001234.5 gene description [organism]
        (r'>(?P<accession>[A-Z]{2}_\d+\.?\d*)\s+(?P<description>.+?)\s*\[(?P<organism>[^\]]+)\]',
         'RefSeq'),
        # Simple: >id description
        (r'>(?P<id>\S+)\s*(?P<description>.*)',
         'Simple'),
    ]
    
    for pattern, fmt in patterns:
        match = re.match(pattern, header)
        if match:
            return {'format': fmt, **match.groupdict()}
    return None


headers = [
    ">gi|12345|gb|ABC123.1| Homo sapiens BRCA1",
    ">sp|P04637|P53_HUMAN Cellular tumor antigen p53 OS=Homo sapiens GN=TP53",
    ">NM_007294.4 Homo sapiens BRCA1 DNA repair [Homo sapiens]",
    ">my_sequence_001 test sequence",
]

for header in headers:
    parsed = parse_fasta_header(header)
    print(f"Header: {header}")
    print(f"Parsed: {parsed}\n")

---
## 10. Bio Application: Parse GenBank Features

In [ ]:
# Simulated GenBank feature table
genbank_features = """     gene            100..500
                     /gene="BRCA1"
                     /locus_tag="b0001"
     CDS             100..500
                     /gene="BRCA1"
                     /protein_id="AAA001.1"
                     /translation="MSLSPADKTN"
     gene            complement(600..900)
                     /gene="TP53"
                     /locus_tag="b0002"
     CDS             join(600..700,750..900)
                     /gene="TP53"
                     /protein_id="AAA002.1"
"""

# Extract gene names
genes = re.findall(r'/gene="([^"]+)"', genbank_features)
print(f"Gene names: {list(set(genes))}")

# Extract protein IDs
protein_ids = re.findall(r'/protein_id="([^"]+)"', genbank_features)
print(f"Protein IDs: {protein_ids}")

# Extract coordinate ranges
coords = re.findall(r'(\d+)\.\.(\d+)', genbank_features)
print(f"\nCoordinate ranges:")
for start, end in coords:
    print(f"  {start}-{end} ({int(end) - int(start) + 1} bp)")

# Find features on complement strand
complement_genes = re.findall(r'complement\((\d+)\.\.(\d+)\)', genbank_features)
print(f"\nComplement strand features: {complement_genes}")

---
## 11. Bio Application: Protein Motifs and PROSITE Patterns

PROSITE uses a special notation for protein motifs:
- `N-{P}-[ST]-{P}` means: N, then not P, then S or T, then not P
- `x` means any amino acid
- `(3)` means repeat 3 times

We can convert these to Python regex.

In [ ]:
def prosite_to_regex(pattern):
    """Convert a PROSITE pattern to a Python regex.
    
    PROSITE syntax:
      - Single letters match themselves
      - [ABC] matches A, B, or C
      - {ABC} matches anything EXCEPT A, B, or C
      - x matches any amino acid
      - (n) means repeat n times
      - (n,m) means repeat n to m times
      - '-' is a separator (ignored)
    """
    result = []
    i = 0
    parts = pattern.replace('-', '')
    
    while i < len(parts):
        if parts[i] == '[':
            end = parts.index(']', i)
            result.append(parts[i:end+1])
            i = end + 1
        elif parts[i] == '{':
            end = parts.index('}', i)
            chars = parts[i+1:end]
            result.append(f'[^{chars}]')
            i = end + 1
        elif parts[i] == 'x':
            result.append('.')
            i += 1
        elif parts[i] == '(':
            end = parts.index(')', i)
            quant = parts[i+1:end]
            if ',' in quant:
                n, m = quant.split(',')
                result.append('{' + n.strip() + ',' + m.strip() + '}')
            else:
                result.append('{' + quant + '}')
            i = end + 1
        else:
            result.append(parts[i])
            i += 1
    
    return ''.join(result)


# N-glycosylation motif: N-{P}-[ST]-{P}
prosite = 'N-{P}-[ST]-{P}'
regex = prosite_to_regex(prosite)
print(f"PROSITE: {prosite}")
print(f"Regex:   {regex}")

In [ ]:
# Search for N-glycosylation sites in a protein
# Human EPO (erythropoietin) has known glycosylation sites
epo_partial = "APPRLICDSRVLERYLLEAKEAENITTGCAEHCSLNENITVPDTKVNFYAWKRMEVGQQAVEVWQGLALLSEAVLRGQALLVNSSQPWEPLQLHVDKAVSGLRSLTTLLRALGAQKEAISPPD"

glyco_pattern = re.compile(prosite_to_regex('N-{P}-[ST]-{P}'))

print(f"Protein ({len(epo_partial)} aa): {epo_partial[:50]}...")
print(f"\nN-glycosylation sites (N-{{P}}-[ST]-{{P}}):")
for m in glyco_pattern.finditer(epo_partial):
    print(f"  Position {m.start()+1}: {m.group()}")

In [ ]:
# More PROSITE patterns
prosite_patterns = {
    'N-glycosylation':     'N-{P}-[ST]-{P}',
    'PKC phosphorylation': '[ST]-x-[RK]',
    'CK2 phosphorylation': '[ST]-x(2)-[DE]',
    'Myristoylation':      'G-{EDRKHPFYW}-x(2)-[STAGCN]-{P}',
}

test_protein = "MNGTSAKNLTCVNSTGMKNQTAPRSNVTKGAISSMSDELLNK"
print(f"Protein: {test_protein}\n")

for name, prosite_pat in prosite_patterns.items():
    regex_pat = prosite_to_regex(prosite_pat)
    matches = [(m.start()+1, m.group()) for m in re.finditer(regex_pat, test_protein)]
    if matches:
        print(f"{name} ({prosite_pat}):")
        for pos, motif in matches:
            print(f"  Position {pos}: {motif}")

---
## 12. Bio Application: Find Restriction Enzyme Sites (EcoRI Example in Detail)

In [ ]:
# EcoRI cuts at G^AATTC (between G and A)
# Let's visualize the cut

def visualize_cuts(sequence, site, cut_offset):
    """Visualize restriction enzyme cuts in a sequence.
    
    cut_offset: position within the recognition site where the cut occurs.
    """
    cut_positions = [m.start() + cut_offset for m in re.finditer(site, sequence)]
    
    print(f"5'- {sequence} -3'")
    
    # Build cut indicator line
    indicator = [' '] * (len(sequence) + 5)
    for pos in cut_positions:
        indicator[pos + 4] = '^'
    print(''.join(indicator))
    
    # Show complement strand
    comp = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}
    comp_seq = ''.join(comp.get(n, n) for n in sequence)
    print(f"3'- {comp_seq} -5'")
    
    return cut_positions


dna = "ATGCGAATTCGATCGATCGAATTCGATCG"
print("EcoRI (G^AATTC) cuts:\n")
cuts = visualize_cuts(dna, 'GAATTC', 1)
print(f"\nCut positions: {cuts}")

---
## 13. Bio Application: Find Palindromic DNA Sequences

A DNA palindrome reads the same on both strands in the 5'-to-3' direction. Restriction enzyme recognition sites are typically palindromic.

In [ ]:
def reverse_complement(seq):
    """Return the reverse complement of a DNA sequence."""
    comp = str.maketrans('ATGC', 'TACG')
    return seq.translate(comp)[::-1]


def find_palindromes(sequence, min_len=4, max_len=8):
    """Find all palindromic subsequences using regex + reverse complement check."""
    palindromes = []
    
    for length in range(min_len, max_len + 1, 2):  # palindromes must be even length
        # Use lookahead to find all overlapping matches of a given length
        for m in re.finditer(f'(?=([ATGC]{{{length}}}))', sequence):
            subseq = m.group(1)
            if subseq == reverse_complement(subseq):
                palindromes.append((m.start(), subseq))
    
    return sorted(palindromes)


dna = "ATGCGAATTCGATGGATCCAAGCTTCATATGATCG"
print(f"Sequence: {dna}\n")

pals = find_palindromes(dna, min_len=4, max_len=8)
print("Palindromic sequences found:")
for pos, seq in pals:
    rc = reverse_complement(seq)
    print(f"  Position {pos:3d}: {seq} (reverse complement: {rc})")

---
## 14. Sequence Validation

In [ ]:
def validate_sequence(sequence, seq_type='dna'):
    """Validate a biological sequence using regex."""
    patterns = {
        'dna':     r'^[ATGCNatgcn]+$',
        'rna':     r'^[AUGCNaugcn]+$',
        'protein': r'^[ACDEFGHIKLMNPQRSTVWYXacdefghiklmnpqrstvwyx]+$',
    }
    
    if seq_type not in patterns:
        raise ValueError(f"Unknown sequence type: {seq_type}")
    
    return bool(re.match(patterns[seq_type], sequence))


test_cases = [
    ("ATGCGATCG",     "dna"),
    ("AUGCGAUCG",     "rna"),
    ("MVLSPADKTNVK",  "protein"),
    ("ATGCXYZ123",    "dna"),
    ("atgcgatcg",     "dna"),
]

print("Sequence validation:")
for seq, stype in test_cases:
    valid = validate_sequence(seq, stype)
    status = 'Valid' if valid else 'INVALID'
    print(f"  {seq:20s} ({stype:7s}): {status}")

---
## 15. Extracting Genomic Coordinates from Text

In [ ]:
# Extract genomic coordinates from free text
text = """
BRCA1 is located at chr17:43044295-43170245 on the minus strand.
TP53 maps to chr17:7,661,779-7,687,550.
EGFR: chromosome 7, positions 55086714 to 55324313.
The MYC oncogene is on chr8:127,735,434-127,742,951.
"""

coord_pattern = re.compile(r'''
    chr(?:omosome\s*)?           # 'chr' or 'chromosome'
    (\d+|[XY])                   # chromosome number
    [:\s,]+                      # separator
    ([\d,]+)                     # start position
    \s*[-to]+\s*                 # dash or 'to'
    ([\d,]+)                     # end position
''', re.VERBOSE | re.IGNORECASE)

print("Extracted genomic coordinates:")
for match in coord_pattern.finditer(text):
    chrom = match.group(1)
    start = int(match.group(2).replace(',', ''))
    end = int(match.group(3).replace(',', ''))
    size = end - start
    print(f"  chr{chrom}:{start:,}-{end:,} ({size:,} bp)")

---
## 16. Finding Tandem Repeats

In [ ]:
def find_tandem_repeats(sequence, min_unit=2, max_unit=6, min_copies=3):
    """Find tandem repeats (microsatellites) in a DNA sequence."""
    results = []
    
    for unit_len in range(min_unit, max_unit + 1):
        # Match a unit repeated min_copies or more times
        pattern = re.compile(r'(([ATGC]{' + str(unit_len) + r'})\2{' + str(min_copies - 1) + r',})')
        
        for m in pattern.finditer(sequence):
            full_repeat = m.group(1)
            unit = m.group(2)
            copies = len(full_repeat) // unit_len
            results.append({
                'position': m.start(),
                'unit': unit,
                'copies': copies,
                'length': len(full_repeat)
            })
    
    return results


# Sequence with microsatellite repeats
dna = "ATGCACACACACACACATGCGATGATGATGATGCTAGCTAGCTAGCTAG"

print(f"Sequence: {dna}\n")
repeats = find_tandem_repeats(dna)

print("Tandem repeats (microsatellites):")
for r in repeats:
    print(f"  Position {r['position']}: ({r['unit']})x{r['copies']} [{r['length']} bp]")

---
## Exercises

### Exercise 1: Extract All ORFs with Regex

Write a function `find_orfs_regex(sequence, min_length=30)` that uses regex to find all open reading frames (ATG ... stop codon) in all three forward reading frames.

Return a list of dictionaries with keys: `frame`, `start`, `end`, `length`, `sequence`.

Test on: `"ATGAAAGCCTTTGGGATCGATCGTAGATGCCCCCCGGGATCGATCGATCGTGA"`

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
def find_orfs_regex(sequence, min_length=30):
    """Find ORFs in all 3 forward reading frames using regex."""
    orf_pattern = re.compile(r'ATG(?:[ATGC]{3})*?(?:TAA|TAG|TGA)')
    orfs = []
    
    for frame in range(3):
        for match in orf_pattern.finditer(sequence[frame:]):
            orf_seq = match.group()
            if len(orf_seq) >= min_length:
                orfs.append({
                    'frame': frame + 1,
                    'start': match.start() + frame,
                    'end': match.end() + frame,
                    'length': len(orf_seq),
                    'sequence': orf_seq
                })
    
    return sorted(orfs, key=lambda x: x['length'], reverse=True)


test_dna = "ATGAAAGCCTTTGGGATCGATCGTAGATGCCCCCCGGGATCGATCGATCGTGA"
orfs = find_orfs_regex(test_dna, min_length=9)

print(f"Sequence ({len(test_dna)} bp): {test_dna}\n")
print("ORFs found:")
for orf in orfs:
    print(f"  Frame +{orf['frame']}: pos {orf['start']}-{orf['end']}, "
          f"{orf['length']} bp: {orf['sequence'][:30]}...")

### Exercise 2: Parse BLAST Tabular Output

BLAST tabular output (`-outfmt 6`) has this format:
```
query_id  subject_id  %identity  align_len  mismatches  gaps  q_start  q_end  s_start  s_end  evalue  bitscore
```

Write a function `parse_blast_line(line)` that uses regex to extract all 12 fields into a dictionary.

Then filter results to keep only hits with identity >= 90% and e-value < 1e-10.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
def parse_blast_line(line):
    """Parse a BLAST tabular output line."""
    pattern = re.compile(
        r'(?P<query>\S+)\t'
        r'(?P<subject>\S+)\t'
        r'(?P<identity>[\d.]+)\t'
        r'(?P<align_len>\d+)\t'
        r'(?P<mismatches>\d+)\t'
        r'(?P<gaps>\d+)\t'
        r'(?P<q_start>\d+)\t'
        r'(?P<q_end>\d+)\t'
        r'(?P<s_start>\d+)\t'
        r'(?P<s_end>\d+)\t'
        r'(?P<evalue>[\d.e-]+)\t'
        r'(?P<bitscore>[\d.]+)'
    )
    match = pattern.match(line)
    if not match:
        return None
    
    result = match.groupdict()
    # Convert numeric fields
    result['identity'] = float(result['identity'])
    result['align_len'] = int(result['align_len'])
    result['mismatches'] = int(result['mismatches'])
    result['gaps'] = int(result['gaps'])
    result['evalue'] = float(result['evalue'])
    result['bitscore'] = float(result['bitscore'])
    return result


# Sample BLAST output
blast_output = """gene1	sp|P04637|P53_HUMAN	98.5	200	3	0	1	200	1	200	1e-50	380.2
gene1	sp|Q8N726|P53_MOUSE	85.2	200	30	0	1	200	1	200	5e-30	250.1
gene2	sp|P38398|BRCA1_HUMAN	99.1	500	4	1	1	500	1	500	0.0	920.5
gene2	sp|P48754|BRCA1_MOUSE	72.3	500	139	5	1	500	1	500	3e-08	180.3
gene3	sp|P00533|EGFR_HUMAN	91.0	300	27	0	1	300	1	300	1e-45	350.0"""

print("All BLAST hits:")
parsed_hits = []
for line in blast_output.strip().split('\n'):
    hit = parse_blast_line(line)
    if hit:
        parsed_hits.append(hit)
        print(f"  {hit['query']} -> {hit['subject']}: {hit['identity']}% (E={hit['evalue']})")

# Filter: identity >= 90% and e-value < 1e-10
filtered = [h for h in parsed_hits if h['identity'] >= 90 and h['evalue'] < 1e-10]
print(f"\nFiltered (identity>=90%, E<1e-10): {len(filtered)} hits")
for h in filtered:
    print(f"  {h['query']} -> {h['subject']}: {h['identity']}% (E={h['evalue']})")

### Exercise 3: Find Palindromic DNA Sequences

Write a function `find_all_palindromes(sequence, length)` that finds all palindromic subsequences of exactly `length` bases. A DNA palindrome is a sequence equal to its reverse complement.

Use regex with lookahead (`(?=(...))`) to find overlapping matches.

Test on: `"ATGCGAATTCGCATGCGATCG"` with length=6.

Hint: GAATTC is palindromic (reverse complement of GAATTC is GAATTC). GCATGC is also palindromic.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
def find_all_palindromes(sequence, length):
    """Find all palindromic subsequences of a given length."""
    comp = str.maketrans('ATGC', 'TACG')
    palindromes = []
    
    # Use lookahead to find overlapping subsequences
    for m in re.finditer(f'(?=([ATGC]{{{length}}}))', sequence):
        subseq = m.group(1)
        rev_comp = subseq.translate(comp)[::-1]
        if subseq == rev_comp:
            palindromes.append((m.start(), subseq))
    
    return palindromes


dna = "ATGCGAATTCGCATGCGATCG"
print(f"Sequence: {dna}\n")

for length in [4, 6]:
    pals = find_all_palindromes(dna, length)
    print(f"Palindromes of length {length}:")
    for pos, seq in pals:
        print(f"  Position {pos}: {seq}")
    print()

### Exercise 4: PROSITE Pattern Matcher

Use the `prosite_to_regex` function to search for the following motifs in the protein sequence below:

1. N-glycosylation: `N-{P}-[ST]-{P}`
2. Casein kinase II phosphorylation: `[ST]-x(2)-[DE]`
3. RGD cell attachment: `R-G-D`

Protein: `"MAANLTKVRGDLPNSTKAFETDMIVNSTGCKSPMAVRGDKNTLQE"`

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
protein = "MAANLTKVRGDLPNSTKAFETDMIVNSTGCKSPMAVRGDKNTLQE"

motifs = {
    'N-glycosylation':     'N-{P}-[ST]-{P}',
    'CK2 phosphorylation': '[ST]-x(2)-[DE]',
    'RGD cell attachment':  'R-G-D',
}

print(f"Protein: {protein}\n")

for name, prosite_pat in motifs.items():
    regex = prosite_to_regex(prosite_pat)
    matches = [(m.start()+1, m.group()) for m in re.finditer(regex, protein)]
    
    print(f"{name} ({prosite_pat} -> {regex}):")
    if matches:
        for pos, seq in matches:
            print(f"  Position {pos}: {seq}")
    else:
        print("  No matches found.")
    print()

### Exercise 5: Multi-format Sequence File Detector

Write a function `detect_format(text)` that uses regex to determine if a text block is:
- **FASTA** (starts with `>`)
- **FASTQ** (starts with `@`, has `+` line, has quality line)
- **GenBank** (starts with `LOCUS`)
- **Unknown**

Test with sample strings.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
def detect_format(text):
    """Detect biological file format from text content."""
    text = text.strip()
    
    # FASTQ: @header\nsequence\n+\nquality
    if re.match(r'^@\S+.*\n[A-Za-z]+\n\+', text, re.MULTILINE):
        return 'FASTQ'
    
    # GenBank: starts with LOCUS
    if re.match(r'^LOCUS\s+', text):
        return 'GenBank'
    
    # FASTA: starts with >
    if re.match(r'^>', text):
        return 'FASTA'
    
    return 'Unknown'


samples = [
    (">gene1\nATGCGATCG", "FASTA sample"),
    ("@SEQ001\nATGCGATC\n+\nIIIIIIII", "FASTQ sample"),
    ("LOCUS       NM_007294  7088 bp  mRNA", "GenBank sample"),
    ("ATGCGATCGATCG", "Raw sequence"),
]

print("Format detection:")
for text, label in samples:
    fmt = detect_format(text)
    print(f"  {label:20s} -> {fmt}")

### Exercise 6: Extract Protein Domains from InterPro-style Annotation

Given InterPro-style annotations, extract all domain names and their coordinate ranges.

```
IPR013783  Immunoglobulin-like_fold  23-95
IPR007110  Immunoglobulin-like_domain  120-190
IPR003599  Ig_sub  200-270
IPR013783  Immunoglobulin-like_fold  280-350
```

Write a regex to parse each line and find which domain types appear more than once.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
from collections import Counter

annotations = """IPR013783  Immunoglobulin-like_fold  23-95
IPR007110  Immunoglobulin-like_domain  120-190
IPR003599  Ig_sub  200-270
IPR013783  Immunoglobulin-like_fold  280-350
IPR007110  Immunoglobulin-like_domain  400-470"""

pattern = re.compile(r'(IPR\d+)\s+(\S+)\s+(\d+)-(\d+)')

domains = []
for match in pattern.finditer(annotations):
    domains.append({
        'ipr_id': match.group(1),
        'name': match.group(2),
        'start': int(match.group(3)),
        'end': int(match.group(4))
    })

print("Parsed domains:")
for d in domains:
    print(f"  {d['ipr_id']} {d['name']:35s} {d['start']}-{d['end']}")

# Find repeated domain types
domain_counts = Counter(d['ipr_id'] for d in domains)
repeated = {ipr: count for ipr, count in domain_counts.items() if count > 1}

print(f"\nRepeated domains:")
for ipr, count in repeated.items():
    name = next(d['name'] for d in domains if d['ipr_id'] == ipr)
    print(f"  {ipr} ({name}): {count} copies")

### Exercise 7: Degenerate Primer Matching

Degenerate primers use IUPAC codes for ambiguous positions. Write a function `iupac_to_regex(primer)` that converts an IUPAC DNA sequence to a regex pattern.

IUPAC ambiguity codes:
- `R` = A or G, `Y` = C or T, `M` = A or C, `K` = G or T
- `S` = G or C, `W` = A or T, `H` = A or C or T, `B` = G or T or C
- `V` = G or C or A, `D` = G or A or T, `N` = any

Then search for the degenerate primer `"ATGRYSWN"` in `"ATGACGCAATGGATCAAATGGCTTGATG"`.

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
IUPAC_MAP = {
    'A': 'A', 'T': 'T', 'G': 'G', 'C': 'C',
    'R': '[AG]', 'Y': '[CT]', 'M': '[AC]', 'K': '[GT]',
    'S': '[GC]', 'W': '[AT]', 'H': '[ACT]', 'B': '[GTC]',
    'V': '[GCA]', 'D': '[GAT]', 'N': '[ATGC]'
}


def iupac_to_regex(primer):
    """Convert an IUPAC DNA primer sequence to a regex pattern."""
    return ''.join(IUPAC_MAP[base] for base in primer.upper())


primer = "ATGRYSWN"
regex = iupac_to_regex(primer)
print(f"Primer (IUPAC): {primer}")
print(f"Regex:          {regex}")

target = "ATGACGCAATGGATCAAATGGCTTGATG"
print(f"\nTarget: {target}")
print("\nMatches:")
for m in re.finditer(regex, target):
    print(f"  Position {m.start()}: {m.group()}")

---
## Summary

### Core Functions
| Function | Returns | Best For |
|---|---|---|
| `re.search()` | First match object or None | Check if pattern exists |
| `re.match()` | Match at start or None | Validate start of string |
| `re.findall()` | List of strings | Get all matches at once |
| `re.finditer()` | Iterator of match objects | Positions + large data |
| `re.sub()` | Modified string | Find and replace |
| `re.compile()` | Compiled pattern | Reuse patterns efficiently |

### Bioinformatics Applications
- **Restriction site mapping**: Find enzyme recognition sequences
- **ORF finding**: Match ATG...stop codon patterns
- **FASTA/GenBank parsing**: Extract gene IDs, coordinates, features
- **Protein motif search**: PROSITE patterns, glycosylation sites
- **Sequence validation**: Check for valid DNA/RNA/protein characters
- **Degenerate primer matching**: IUPAC ambiguity codes
- **Tandem repeat detection**: Microsatellite finding

---

[← Previous: Iterators and Generators for Bioinformatics](../../Tier_1_Python_for_Bioinformatics/11_Iterators_and_Generators/01_iterators_and_generators.ipynb) | [Next: Module 13: Classes and Object-Oriented Programming →](../../Tier_1_Python_for_Bioinformatics/13_Classes_and_OOP/01_classes_and_oop.ipynb)